# Generate a LaTeX Reviewer-Response Template

Edit the metadata, authors, affiliations, and reviewer-comment counts below. Then run all cells to generate `response_to_reviewers_template.tex`.


In [ ]:
# ---------- User inputs ----------

manuscript_title = "Manuscript Title Here"
journal_name = "Journal Name Here"
submission_id = "XXXX"
corresponding_author_email = "email@domain.com"

authors = [
    {"name": "Author One", "affils": [1], "corresponding": True},
    {"name": "Author Two", "affils": [2], "corresponding": False},
    {"name": "Author Three", "affils": [3], "corresponding": False},
]

affiliations = {
    1: "Affiliation One",
    2: "Affiliation Two",
    3: "Affiliation Three",
}

# Number of placeholder comments for each reviewer
reviewer_comment_counts = {
    1: 2,
    2: 2,
    3: 2,
}

output_tex_file = "response_to_reviewers_template.tex"

In [ ]:
def latex_escape(text):
    """Escape common LaTeX-sensitive characters in plain metadata fields."""
    replacements = {
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(ch, ch) for ch in str(text))


def format_affil_list(affils):
    return ",".join(str(a) for a in affils)


def build_author_block(authors, affiliations, corresponding_author_email):
    lines = []

    for author in authors:
        name = latex_escape(author["name"])
        affil_tag = format_affil_list(author["affils"])

        if author.get("corresponding", False):
            lines.append(
                rf"\author[{affil_tag}]{{{name}\thanks{{corresponding author: {latex_escape(corresponding_author_email)}}}}}"
            )
        else:
            lines.append(rf"\author[{affil_tag}]{{{name}}}")

    lines.append("")

    for idx, affil in affiliations.items():
        lines.append(rf"\affil[{idx}]{{{latex_escape(affil)}}}")

    return "\n".join(lines)


def build_reviewer_section(reviewer_number, n_comments):
    section = [
        rf"\section*{{Reviewer \#{reviewer_number}}}",
        "",
        rf"\begin{{enumerate}}[label=\textbf{{Comment {reviewer_number}.\arabic*:}}]",
        "",
    ]

    for _ in range(n_comments):
        section.extend([
            r"\item",
            r"\commenttext{Paste reviewer comment here.}",
            "",
            r"\responsetext{Write the response here.}",
            "",
            r"\revisiontext{Describe the manuscript change here. For example: The relevant text has been revised in Lines XX--YY.}",
            "",
        ])

    section.extend([
        r"\end{enumerate}",
        "",
    ])

    return "\n".join(section)

In [ ]:
latex_template = r"""\documentclass[10pt]{article}

%%%%% math
\usepackage[centertags]{amsmath}
\usepackage{amssymb}
\usepackage{amsfonts}
\usepackage{bm}
\usepackage{upgreek}
\usepackage{skmath}
\usepackage{authblk}

%%%%% graphics
\usepackage[]{graphicx}
\usepackage[dvipsnames]{xcolor}
\usepackage[tight,nice]{units}

%%%%% formatting
\usepackage{enumitem}
\setlist[enumerate]{leftmargin=*,partopsep=0pt,itemsep=6pt,parsep=6pt}
\usepackage[normalem]{ulem}
\usepackage[margin=1.25in]{geometry}
\usepackage{setspace}
\usepackage[labelfont=bf,labelsep=space]{caption}

%%%%% colors and links
\definecolor{Blue}{rgb}{0,0,.8}
\definecolor{Green}{rgb}{0,.5,0}
\definecolor{Gray}{rgb}{.25,.25,.25}
\usepackage[colorlinks=true,citecolor=Green,linkcolor=Black,urlcolor=Blue,bookmarks=false]{hyperref}

%%%%% citations
\usepackage[sort]{natbib}
\setlength{\bibsep}{2pt}
\bibliographystyle{abbrv}

%%%%% footnotes with symbols
\usepackage{perpage}
\MakePerPage{footnote}
\renewcommand{\thefootnote}{\fnsymbol{footnote}}

%%%%%% Math Environment
\renewcommand{\vec}[1]{\mathbf{#1}}
\newcommand{\mat}[1]{\mathsf{#1}}

%%%%% floats
\setlength{\textfloatsep}{0pt}
\setlength{\belowcaptionskip}{11pt}
\renewcommand{\topfraction}{0.9}
\renewcommand{\dbltopfraction}{0.9}
\renewcommand{\textfraction}{0.07}
\renewcommand{\floatpagefraction}{0.85}
\renewcommand{\dblfloatpagefraction}{0.9}

%%%%% graphics path
\graphicspath{{./figures/}}

%%%%% paragraphs
\setlength{\parindent}{0pt}
\setlength{\parskip}{6pt}

%%%%% quote indent
\renewenvironment{quote}{\color{Blue}\list{}{\leftmargin=.25in\rightmargin=0.25in\topsep=0in\parsep=6pt}\item[]}{\endlist}

%%%%% response commands
\newcommand{\commenttext}[1]{%
\noindent\textbf{Reviewer comment:}
\begin{quote}
#1
\end{quote}
}

\newcommand{\responsetext}[1]{%
\noindent\textbf{Response:} #1
}

\newcommand{\revisiontext}[1]{%
\noindent\textbf{Revision made:} #1
}

%%%%% title information
\title{\large \bf Response to Reviewers' Comments:\\[2pt]
\vspace{5pt}
\Large __MANUSCRIPT_TITLE__\\
\vspace{5pt}
\emph{__JOURNAL_NAME__}\\
\vspace{5pt}
\large Submission~ID:~__SUBMISSION_ID__}

__AUTHOR_BLOCK__

\date{\today\\[2\baselineskip]\small{}}

\begin{document}

\maketitle

\begin{quote}
We thank the editor and reviewers for their time and for their constructive comments. We have revised the manuscript in response to the comments. The reviewer comments are reproduced below in blue, followed by our responses and a summary of the corresponding revisions. Line numbers refer to the revised manuscript.
\end{quote}

__REVIEWER_SECTIONS__

\section*{Additional Changes}

Briefly list any additional revisions that were not tied to a specific reviewer comment. If there are no such changes, this section can be removed.

\end{document}
"""

In [ ]:
author_block = build_author_block(
    authors=authors,
    affiliations=affiliations,
    corresponding_author_email=corresponding_author_email,
)

reviewer_sections = "\n".join(
    build_reviewer_section(reviewer_number, n_comments)
    for reviewer_number, n_comments in reviewer_comment_counts.items()
)

latex_code = (
    latex_template
    .replace("__MANUSCRIPT_TITLE__", latex_escape(manuscript_title))
    .replace("__JOURNAL_NAME__", latex_escape(journal_name))
    .replace("__SUBMISSION_ID__", latex_escape(submission_id))
    .replace("__AUTHOR_BLOCK__", author_block)
    .replace("__REVIEWER_SECTIONS__", reviewer_sections)
)

with open(output_tex_file, "w", encoding="utf-8") as f:
    f.write(latex_code)

print(f"LaTeX template written to: {output_tex_file}")

In [ ]:
# Optional: preview the first part of the generated LaTeX code
print(latex_code[:2000])